In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re

In [37]:
df = pd.read_csv('../data/preprocessed_data/news_data.csv')
print(df.shape)
df.head()


(58, 8)


,company,ticker,detected_name,text,nid,article_title,article_url,date
0,Maybank,1155.KL,Malayan Banking Bhd (Maybank),Malayan Banking Bhd (Maybank) has dismissed al...,604978,"Luster Industries, Aimflex, Opcom, TM, Maybank...",https://theedgemalaysia.com/article/luster-ind...,2022-01-24 14:32:04
1,CIMB,1023.KL,CIMB Group Holdings Bhd,CIMB Group Holdings Bhd ’s (KL: CIMB ) net pro...,816234,"Yinson, Gamuda, WCT, Zetrix AI, HeiTech Padu, ...",https://theedgemalaysia.com/node/816234,2026-08-28 15:49:34
2,CIMB,1023.KL,CIMB Group Holdings Bhd,CIMB Group Holdings Bhd (KL: CIMB ) posted a s...,805091,"Berjaya, Hong Leong Bank, UOA Development, Hon...",https://theedgemalaysia.com/node/805091,2026-05-26 15:54:50
3,CIMB,1023.KL,CIMB Group Holdings Bhd,CIMB Group Holdings Bhd (KL: CIMB ) has appoin...,802771,"SD Guthrie, Sime Darby, CIMB, AirAsia X, Lotte...",https://theedgemalaysia.com/node/802771,2026-05-07 14:57:58
4,CIMB,1023.KL,CIMB Group Holdings Bhd,CIMB Group Holdings Bhd (KL: CIMB ) is selling...,802431,"Public Bank, CIMB Group, Gamuda, NexG, Telekom...",https://theedgemalaysia.com/node/802431,2026-05-05 15:26:43


In [38]:
display(df.info())
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   company        58 non-null     object
 1   ticker         58 non-null     object
 2   detected_name  58 non-null     object
 3   text           58 non-null     object
 4   nid            58 non-null     int64 
 5   article_title  58 non-null     object
 6   article_url    58 non-null     object
 7   date           58 non-null     object
dtypes: int64(1), object(7)
memory usage: 3.8+ KB


None

,nid
count,58.000000
mean,748965.327586
std,50052.945701
min,604978.000000
25%,717895.750000
50%,752305.500000
75%,799466.000000
max,816234.000000


In [ ]:
from transformers import pipeline
finbert = pipeline("sentiment-analysis",model="ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [63]:
def analyze_sentiment(text):

    text = str(text)
    text = " ".join(text.split())

    results = finbert(
        text,
        top_k=None,
        truncation=True
    )

    scores = {
        item["label"]: item["score"]
        for item in results
    }

    sentiment = max(scores,key=scores.get)

    sentiment_score = (
        scores["positive"]
        - scores["negative"]
    )

    return {
        "sentiment": sentiment,
        "positive_score": scores["positive"],
        "negative_score": scores["negative"],
        "neutral_score": scores["neutral"],
        "sentiment_score": sentiment_score
    }

In [ ]:
from tqdm import tqdm
for index, row in tqdm(df.iterrows()):
    text = row["text"]
    sentiment_results = analyze_sentiment(text)
    df.at[index, "sentiment"] = sentiment_results["sentiment"]
    df.at[index, "positive_score"] = sentiment_results["positive_score"]
    df.at[index, "negative_score"] = sentiment_results["negative_score"]
    df.at[index, "neutral_score"] = sentiment_results["neutral_score"]
    df.at[index, "sentiment_score"] = sentiment_results["sentiment_score"]
    

58it [00:06,  9.51it/s]


In [66]:
df[
    [
        "company",
        "sentiment",
        "positive_score",
        "negative_score",
        "neutral_score",
        "sentiment_score"
    ]
].head()

,company,sentiment,positive_score,negative_score,neutral_score,sentiment_score
0,Maybank,neutral,0.155793,0.228810,0.615397,-0.073017
1,CIMB,positive,0.567061,0.396734,0.036205,0.170326
2,CIMB,negative,0.035997,0.953414,0.010590,-0.917417
3,CIMB,neutral,0.027536,0.032356,0.940108,-0.004821
4,CIMB,neutral,0.058317,0.014921,0.926762,0.043396


In [68]:
df["sentiment"].value_counts(normalize=True) * 100

sentiment
positive    51.724138
negative    29.310345
neutral     18.965517
Name: proportion, dtype: float64

In [69]:
pd.crosstab(
    df["company"],
    df["sentiment"]
)

sentiment,negative,neutral,positive
company,,,
CIMB,5,7,14
Maybank,0,1,0
Petronas Chemicals,3,0,5
Public Bank,7,3,7
Tenaga Nasional,2,0,4


In [70]:
df.groupby("company")["sentiment_score"].mean().sort_values(
    ascending=False
)

company
Tenaga Nasional       0.283854
CIMB                  0.237116
Petronas Chemicals    0.133486
Public Bank           0.020290
Maybank              -0.073017
Name: sentiment_score, dtype: float64

In [71]:
df.to_csv(
    "../data/preprocessed_data/news_data_sentiment.csv",
    index=False
)

df.to_parquet(
    "../data/preprocessed_data/news_data_sentiment.parquet",
    index=False
)